# Crimes Database

### TOC

### Intro

In this guided project, we will demonstrate the fundamentals of PostgreSQL by building a functional database. The structure will include a defined schema and a primary table, which we will populate by loading data from a `data.csv` file. Additionally, we will illustrate how to define two distinct user groups and assign specific users to each. 

**Rough Project Overview:**

![project overview](project-overview.png)

### Setup

To begin, we will establish a connection using the specific Postgres user created for this project. Once connected, we will execute the commands necessary to initialize our database and schema.

In [1]:
#import psycopg2

#conn = psycopg2.connect(
#    dbname="dq", 
#    user="dq",         # user created for project purposes only
#    password="abc123",
#    host="localhost"
#)
#conn.autocommit = True
#cur = conn.cursor()
#cur.execute("CREATE DATABASE crimes_db;")
#conn.close()

In [2]:
#conn = psycopg2.connect(
#    dbname="crimes_db", 
#    user="dq",         # user created for project purposes only
#    password="abc123",
#    host="localhost"
#)
#conn.autocommit = True
#cur = conn.cursor()
#cur.execute("CREATE SCHEMA crimes;")

### Data

With the database and schema prepared, we are now ready to process our information. We will begin by using the `csv` module to import and read the raw data.

In [3]:
import csv
with open('boston.csv') as file:
    reader = csv.reader(file)
    col_headers = next(reader)
    first_row = next(reader)

print(col_headers)
print(first_row)

['incident_number', 'offense_code', 'description', 'date', 'day_of_the_week', 'lat', 'long']
['1', '619', 'LARCENY ALL OTHERS', '2018-09-02', 'Sunday', '42.35779134', '-71.13937053']


### Data Types

Before migrating the data into our schema’s table, we will analyze the data types of each column to ensure the database is as optimized as possible. To facilitate this, we will define a custom function that compiles a set of all unique values within a specific column. By examining these unique values, we can accurately determine the most appropriate data type for every field.

In [4]:
def get_col_set(csv_file, col_index):
    values = set()
    with open(csv_file, 'r') as f:
        next(f)
        reader = csv.reader(f)
        for row in reader:
            values.add(row[col_index])
    return values

for i in range(len(col_headers)):
    values = get_col_set("boston.csv", i)
    print(col_headers[i], len(values), sep='\t')

incident_number	298329
offense_code	219
description	239
date	1177
day_of_the_week	7
lat	18177
long	18177


Having identified the number of unique values in each column, we will now focus on the two columns containing textual data: `description` and `day_of_the_week`. Our objective is to identify the longest possible string in these columns to determine the most efficient data types. 

The `day_of_the_week` column contains only seven unique values, representing the days of the week; therefore, "Wednesday" is the longest possible string. The `description` column, however, contains 239 unique values. To find the maximum string length for this column, we will use our custom `get_col_set()` function to generate a set of all unique strings and then compute the length of the longest entry.

In [5]:
descriptions = get_col_set("boston.csv", 2)
longest_string = max(descriptions, key=len) 
print(longest_string)
print(len(longest_string))

RECOVERED - MV RECOVERED IN BOSTON (STOLEN OUTSIDE BOSTON)
58


### Column Data Type Selection

Based on the analysis above, we can now assign the most appropriate data types to our table columns. As a reminder, our data includes the following fields:

* `incident_number`
* `offense_code`
* `description`
* `date`
* `day_of_the_week`
* `lat`
* `long`

We will assign the `INTEGER` type to `incident_number` and `offense_code` since they contain numeric data. For the `description` column, we will use `VARCHAR(100)`, which provides ample space for the longest strings while remaining memory-efficient. The `date` column will use the `DATE` type, while `lat` and `long` will be stored as `DECIMAL`. Finally, because the `day_of_the_week` column contains a small, fixed set of values, we will use an `ENUM` type for better organization and performance.

**Summary of Data Type Assignments:**

| Column Name | Data Type |
| :--- | :--- |
| `incident_number` | INTEGER |
| `offense_code` | INTEGER |
| `description` | VARCHAR(100) |
| `date` | DATE |
| `day_of_the_week` | ENUM |
| `lat` | DECIMAL |
| `long` | DECIMAL |

We will proceed by creating the table and defining these data types in the code block below.

In [6]:
# Create the enumerated datatype for representing the weekday
# cur.execute("""
#     CREATE TYPE weekday AS ENUM ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday');
# """)

# Create the table
# cur.execute("""
#     CREATE TABLE crimes.boston_crimes (
#         incident_number INTEGER PRIMARY KEY,
#         offense_code INTEGER,
#         description VARCHAR(100),
#         date DATE,
#         day_of_the_week weekday,
#         lat decimal,
#         long decimal
#     );
# """)

### Data Migration

With our table prepared inside our schema, we are now ready to load our data into it.

In [7]:
# loading the data into our boston_crimes table
# with open("boston.csv") as f:
#     cur.copy_expert("COPY crimes.boston_crimes FROM STDIN WITH CSV HEADER;", f)

# ensuring the data was loaded successfully by printing the number of rows
# cur.execute("SELECT * FROM crimes.boston_crimes")
# print(len(cur.fetchall()))

### Users

Now that our database is fully created, complete with a schema and table, we are ready to move onto creating our user groups and users. We want to impliment the least privilege principle by creating two groups of users with specific permissions. Once we have created our groups, we will create and assign one user to each group.

To begin this prosess, we will first revoke all public privilages to our database. This will ensure that no users mistakenly have more permissions than intended. Once we revoke all privilages, we can reassign the proper permissions to the appropriate groups and users.

In [8]:
# cur.execute("REVOKE ALL ON SCHEMA public FROM public;")
# cur.execute("REVOKE ALL ON DATABASE crime_db FROM public;")

With our database protected from users with unintended privlieges, we will proceed to make our first user group. This first group will have permission to read the database only, appropriate for users along the lines of a Data Analyist.

In [9]:
# cur.execute("CREATE GROUP readonly NOLOGIN;")
# cur.execute("GRANT CONNECT ON DATABASE crime_db TO readonly;")
# cur.execute("GRANT USAGE ON SCHEMA crimes TO readonly;")
# cur.execute("GRANT SELECT ON ALL TABLES IN SCHEMA crimes TO readonly;")

Our second user group will have permission to both read and write in the database, appropriate for Data Science type users.

In [10]:
# cur.execute("CREATE GROUP readwrite NOLOGIN;")
# cur.execute("GRANT CONNECT ON DATABASE crime_db TO readwrite;")
# cur.execute("GRANT USAGE ON SCHEMA crimes TO readwrite;")
# cur.execute("GRANT SELECT, INSERT, DELETE, UPDATE ON ALL TABLES IN SCHEMA crimes TO readwrite;")

Having created both our groups and assigned their privilages, we will now create one user for each group. To the readonly group, we will assign the data_analyst user, and to the readwrite group we will assign the data_scientist user.

In [11]:
# cur.execute("CREATE USER data_analyst WITH PASSWORD 'secret1';")
# cur.execute("GRANT readonly TO data_analyst;")

# cur.execute("CREATE USER data_scientist WITH PASSWORD 'secret2';")
# cur.execute("GRANT readwrite TO data_scientist;")

### Verification and Conclusion

As our final section of this project, we want to quickly verify the execution of our groups and users was successful. We will do so by testing that our groups are indeed assigned the correct privilages, by examining the `information_schema.table_privileges` table.

In [ ]:
# cur.execute("""
#     SELECT grantee, privilege_type
#     FROM information_schema.table_privileges
#     WHERE grantee = 'readwrite';
# """)

# cur.execute("""
#     SELECT grantee, privilege_type
#     FROM information_schema.table_privileges
#     WHERE grantee = 'readonly';
# """)